# MJX 07 — GPU RL with MuJoCo Playground (PPO control)

### Lab Description

We train a control policy with **PPO** on the GPU using **MuJoCo Playground** (DeepMind's GPU-accelerated robot-learning library, built on MJX). The task is `CartpoleBalance` from the DM Control Suite — a classic benchmark that learns in a couple of minutes on the AMD GPU.

**Stable settings for this ROCm stack** (gfx1151 APU). Playground imports cleanly on the **torch-free** `auplc-mujoco-mjx` image. A few knobs keep training healthy:
- **`num_envs = 512`** — the largest batch that is both crash-free and numerically stable here: `2048` trips a GPU memory-aperture fault and `1024` diverges to NaN on the first update, while `512` converges smoothly to ~1000 and holds.
- **`jax_default_matmul_precision = "highest"`** — avoids fp cast overflow.
- **`learning_rate = 3e-4`** (below the `1e-3` default) — at this batch the default rate diverges; the lower rate climbs smoothly and **stays** at the top, so the final weights are already good (no checkpoint cherry-picking).
- Physics is forced to the pure-JAX path (`impl="jax"`) because the `mujoco_warp` kernels are CUDA-only.

#### Recommended Hardware
AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)
#### Software Environment
OS: Ubuntu 24.04.4 LTS \
Install [AUP learning cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=max-pro-395). After installing AUP Learning Cloud you will have the ROCm + JAX/MJX environment (the `auplc-mujoco-mjx` course image) that this notebook is built for.

## Goals
- Train a PPO policy on a Playground task on the AMD GPU
- Read a learning curve and recognize convergence
- Understand the ROCm-specific stability settings
- Compare an untrained vs. trained policy visually

### Import libraries and apply the compatibility shim

Brax 0.14 calls `jax.device_put_replicated`, which the jax-rocm build removed; we re-add a single-device version. We also set the highest matmul precision to avoid fp overflow during training.

In [ ]:
import os, functools
os.environ["MUJOCO_GL"] = "egl"

import jax
import jax.numpy as jp
# Brax 0.14 calls jax.device_put_replicated, removed in the jax 0.10 that ships
# with jax-rocm. Re-add a single-device version.
if not hasattr(jax, "device_put_replicated"):
    def _dpr(x, devices=None):
        n = len(devices) if devices is not None else jax.local_device_count()
        return jax.tree_util.tree_map(
            lambda a: jax.device_put(jp.broadcast_to(jp.asarray(a)[None], (n,) + jp.asarray(a).shape)), x)
    jax.device_put_replicated = _dpr
# Use the highest matmul precision to avoid fp overflow during training.
jax.config.update("jax_default_matmul_precision", "highest")

import numpy as np
import matplotlib.pyplot as plt
import imageio
from mujoco_playground import registry, wrapper
from mujoco_playground.config import dm_control_suite_params
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks
from IPython.display import Video

print("JAX devices:", jax.devices())

### Load the task and build the PPO config

We load `CartpoleBalance` (pure-JAX physics) and start from Playground's tuned PPO config, then override the batch size, learning rate, and horizon with the ROCm-stable values described above.

In [ ]:
ENV_NAME = "CartpoleBalance"
env = registry.load(ENV_NAME, config_overrides={"impl": "jax"})

# Start from Playground's tuned PPO config, then shrink batch + horizon so the
# notebook trains quickly and safely on this GPU.
cfg = dm_control_suite_params.brax_ppo_config(ENV_NAME)
ppo_params = cfg.to_dict()
net_cfg = ppo_params.pop("network_factory", None)
ppo_params.update(
    # NOTE on step counts: brax trains in whole "eval intervals" and rounds
    # num_timesteps UP to (num_evals - 1) of them. For this config one interval is
    # ~1,228,800 env steps, so the run always executes (num_evals - 1) * 1,228,800.
    # We set num_timesteps to exactly that value so the logged step count matches
    # the code: 6 * 1,228,800 = 7,372,800.
    num_evals=7,
    num_timesteps=7_372_800,   # = 6 eval intervals; the log ends at this exact number
    num_envs=512,
    batch_size=512,
    num_minibatches=8,
    learning_rate=3e-4,  # below the 1e-3 default: at this batch the high rate diverges to NaN
)
print({k: ppo_params[k] for k in ["num_timesteps", "num_envs", "batch_size", "episode_length"]})

### Train

`ppo.train` runs the full PPO loop on the GPU: collect rollouts from 512 parallel envs, compute advantages, update the policy/value networks, and evaluate periodically. The `progress_fn` prints the eval reward at each checkpoint — watch it climb toward ~1000 and stay there.

In [ ]:
progress = []
def progress_fn(step, metrics):
    r = float(metrics.get("eval/episode_reward", 0.0))
    progress.append((int(step), r))
    print(f"step {int(step):>9}  eval reward {r:8.1f}")

train_fn = functools.partial(ppo.train, **ppo_params)
if net_cfg:
    train_fn = functools.partial(train_fn,
        network_factory=functools.partial(ppo_networks.make_ppo_networks, **net_cfg))

# We use the FINAL trained weights directly: at this learning rate the run
# converges and stays near the top, so no checkpoint cherry-picking is needed.
make_inference_fn, params, _ = train_fn(
    environment=env,
    wrap_env_fn=wrapper.wrap_for_brax_training,
    progress_fn=progress_fn,
    seed=0,
)
print(f"training done — final eval reward {progress[-1][1]:.1f}")

### Plot the learning curve

A healthy run rises and then plateaus near the maximum return (~1000 = balanced for the full episode).

In [ ]:
steps, rewards = zip(*progress)
plt.figure(figsize=(7, 3))
plt.plot(steps, rewards, marker="o")
plt.xlabel("environment steps"); plt.ylabel("eval episode reward")
plt.title(f"PPO learning curve ({ENV_NAME})"); plt.grid(True); plt.show()

### Did it actually learn? — random vs. trained

The curve says yes; now let's *see* it. We render two rollouts side by side for the same 200 steps. **Left: random actions** (the pole tips over and the cart drifts). **Right: the trained PPO policy** (the pole is held upright).

In [ ]:
# Compare an UNTRAINED (random) policy with the TRAINED PPO policy.
os.makedirs("output/videos", exist_ok=True)
inference = jax.jit(make_inference_fn(params))
reset, step = jax.jit(env.reset), jax.jit(env.step)

def rollout(action_fn, n=200, seed=1):
    rng = jax.random.PRNGKey(seed)
    state = reset(rng)
    traj = [state]
    for _ in range(n):  # no early stop, so a failing policy is fully visible
        rng, k = jax.random.split(rng)
        state = step(state, action_fn(state.obs, k))
        traj.append(state)
    return traj

# "Not learned": random actions in the action space.
random_fn  = lambda obs, k: jax.random.uniform(k, (env.action_size,), minval=-1.0, maxval=1.0)
# "Learned": the trained PPO policy.
trained_fn = lambda obs, k: inference(obs, k)[0]

f_rand    = np.asarray(env.render(rollout(random_fn),  height=240, width=320))
f_trained = np.asarray(env.render(rollout(trained_fn), height=240, width=320))
sep = np.full((f_rand.shape[0], f_rand.shape[1], 4, 3), 255, np.uint8)  # white divider
combo = np.concatenate([f_rand, sep, f_trained], axis=2)  # left = random, right = trained
out = "output/videos/mjx07_random_vs_trained.mp4"
imageio.mimsave(out, list(combo), fps=30)
print("saved", combo.shape[0], "frames ->", out)

### Watch the comparison

In [ ]:
Video(url="output/videos/mjx07_random_vs_trained.mp4")

## Conclusions

PPO learned to balance the cartpole from scratch on the AMD GPU, and the trained policy clearly beats random actions. You also met the ROCm-specific knobs (`num_envs=512`, lower learning rate, highest matmul precision) that keep training stable. MJX 08 applies the same recipe to a second Playground task (PointMass).

---

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT